# ChessGPT: Transformer-based AlphaZero
This notebook trains a decision transformer on chess moves.

In [ ]:
!pip install python-chess torch numpy tqdm

In [ ]:
import torch
print(torch.cuda.is_available())

## 1. Utils

In [ ]:
%%writefile utils.py
import chess
import numpy as np
import torch

# Configuration
# Configuration
board_size_sq = 64
CONTEXT_SIZE = 64 # Fixed for board state (8x8)
# Tokens: 0=Empty, 1-6=White, 7-12=Black, 13=PAD?

def encode_move(move):
    return move.from_square * 64 + move.to_square

def decode_move(index, board):
    from_sq = index // 64
    to_sq = index % 64
    move = chess.Move(from_sq, to_sq)
    # Auto-promote to Queen for simplicity in decoding
    if chess.square_rank(to_sq) in [0, 7]:
        p = board.piece_at(from_sq)
        if p and p.piece_type == chess.PAWN:
            move.promotion = chess.QUEEN
    return move

def board_to_sequence(board):
    # Map pieces to integers
    # Empty=0
    # White: P=1, N=2, B=3, R=4, Q=5, K=6
    # Black: p=7, n=8, b=9, r=10, q=11, k=12
    # We can use piece.piece_type (1..6) and color
    # White: type
    # Black: type + 6
    
    seq = []
    # Always consistent order: A1..H8 (square 0..63)
    for sq in range(64):
        p = board.piece_at(sq)
        if p is None:
            seq.append(0)
        else:
            val = p.piece_type # 1..6
            if p.color == chess.BLACK:
                val += 6
            seq.append(val)
    return seq

def sequence_to_tensor(sequence, device='cuda'):
    # sequence is list of 64 ints
    return torch.tensor([sequence], dtype=torch.long, device=device)

def get_material_score(board):
    # Standard values: P=1, N=3, B=3, R=5, Q=9
    values = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 0
    }
    
    score = 0
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece:
            val = values.get(piece.piece_type, 0)
            if piece.color == chess.WHITE:
                score += val
            else:
                score -= val
    return score



## 2. Transformer Model

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ChessTransformer(nn.Module):
    def __init__(self, vocab_size=16, d_model=256, nhead=8, num_layers=6, max_len=64):
        super(ChessTransformer, self).__init__()
        # vocab_size = 13 (0..12) + margin = 16
        # max_len = 64 squares
        self.d_model = d_model
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        # Positional encoding is crucial for it to know A1 vs H8
        self.pos_encoder = nn.Parameter(torch.zeros(1, max_len, d_model))
        
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=512, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        
        # Policy Head now needs to pool the board state?
        # Standard BERT/ViT approach: Use the [CLS] token or Flatten.
        # OR: We can just use the output of the whole sequence flattened?
        # 64 * 256 = 16384 -> 4096 is big but doable.
        # Alternatively, simpler: MaxPool or MeanPool over the 64 squares.
        
        # Let's use Flatten -> Linear for the Policy. It lets it see the whole board relation.
        self.policy_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(max_len * d_model, 4096)
        )
        
        # Value Head
        self.value_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(max_len * d_model, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Tanh()
        )

    def forward(self, x):
        # x: [batch, 64]
        batch, seq_len = x.size()
        
        x = self.embedding(x) * math.sqrt(self.d_model)
        x += self.pos_encoder[:, :seq_len, :]
        
        x = self.transformer_encoder(x)
        
        # x is [batch, 64, d_model]
        # We process the WHOLE board state to decide the move.
        
        policy_logits = self.policy_head(x)
        value = self.value_head(x)
        
        return policy_logits, value


## 3. MCTS

In [ ]:
%%writefile mcts.py
import math
import numpy as np
import torch
import chess
from utils import board_to_sequence, encode_move, CONTEXT_SIZE, get_material_score

class MCTSNode:
    def __init__(self, board, parent=None, prior=0):
        self.board = board
        self.parent = parent
        self.children = {}
        self.visit_count = 0
        self.value_sum = 0
        self.prior = prior
        self.is_expanded = False
        
    def value(self):
        if self.visit_count == 0:
            return 0
        return self.value_sum / self.visit_count

class MCTS:
    def __init__(self, model, device='cuda', c_puct=1.0):
        self.model = model
        self.device = device
        self.c_puct = c_puct
        
    def search(self, board, simulations=50):
        root = MCTSNode(board.copy(), prior=0)
        
        # Expand root
        self._expand(root)
        
        for _ in range(simulations):
            node = root
            while node.is_expanded and len(node.children) > 0:
                node = self._select_child(node)
                
            value = self._expand(node)
            self._backpropagate(node, value)
            
        return root

    def _select_child(self, node):
        best_score = -float('inf')
        best_child = None
        total_visits = sum(child.visit_count for child in node.children.values())
        sqrt_total_visits = math.sqrt(total_visits)
        
        for move_idx, child in node.children.items():
            q_value = -child.value()
            u_value = self.c_puct * child.prior * sqrt_total_visits / (1 + child.visit_count)
            score = q_value + u_value
            if score > best_score:
                best_score = score
                best_child = child
        return best_child

    def _expand(self, node):
        board = node.board
        if board.is_game_over():
            result = board.result()
            if result == '1-0': return 1 if board.turn == chess.WHITE else -1
            elif result == '0-1': return -1 if board.turn == chess.WHITE else 1
            else: return 0

        # TRANSFORMER INPUT
        # Convert board -> Sequence (64 tokens)
        sequence = board_to_sequence(board)
        seq_tensor = torch.tensor([sequence], dtype=torch.long, device=self.device)
        
        self.model.eval()
        with torch.no_grad():
            policy_logits, value = self.model(seq_tensor)
            
        policy_probs = torch.softmax(policy_logits, dim=1).cpu().numpy()[0]
        
        # HYBRID EVALUATION: Blend Learned Value with Material Heuristic
        # This helps the model "know" about material immediately during the game
        # without waiting for thousands of games to learn it.
        
        nn_value = value.item() # [-1, 1], Relative to Current Player
        
        # Calculate Material Score (e.g. +3, -5)
        # get_material_score returns (White - Black)
        mat_score = get_material_score(board) 
        
        # CRITICAL FIX: MCTS expects value "For Current Player".
        # If I am Black, positive material for White is BAD for me.
        if board.turn == chess.BLACK:
            mat_score = -mat_score
            
        mat_value = np.tanh(mat_score / 3.0) # Relative Material Advantage
        
        # Weighted mix: 30% Brain, 70% Material Rule
        # We assume the untrained brain is noisy, so we trust Material more initially.
        nodes_value = (0.3 * nn_value) + (0.7 * mat_value)
        
        policy_sum = 0
        legal_moves = list(board.legal_moves)
        for move in legal_moves:
            move_idx = encode_move(move)
            if move_idx < 4096:
                prob = policy_probs[move_idx]
                policy_sum += prob
                next_board = board.copy()
                next_board.push(move)
                child = MCTSNode(next_board, parent=node, prior=prob)
                node.children[move_idx] = child
        
        if policy_sum > 0:
            for child in node.children.values():
                child.prior /= policy_sum
                
        node.is_expanded = True
        return nodes_value

    def _backpropagate(self, node, value):
        curr = node
        while curr is not None:
            curr.visit_count += 1
            curr.value_sum += value
            curr = curr.parent
            value = -value

    def get_action_probs(self, root, temperature=1.0):
        visits = [child.visit_count for child in root.children.values()]
        moves = [move_idx for move_idx in root.children.keys()]
        if sum(visits) == 0: return [], []
        
        if temperature == 0:
            best_idx = np.argmax(visits)
            probs = [0] * len(visits)
            probs[best_idx] = 1
            return moves, probs
        
        scaled_visits = [v ** (1.0 / temperature) for v in visits]
        sum_scaled = sum(scaled_visits)
        probs = [v / sum_scaled for v in scaled_visits]
        return moves, probs


## 4. Trainer

In [ ]:
%%writefile trainer.py
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import sys
import chess
import chess.svg
from tqdm import tqdm
import os
import random

from model import ChessTransformer
from mcts import MCTS
from utils import board_to_sequence, encode_move, CONTEXT_SIZE, get_material_score

class ChessDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        # example: (sequence, policy_dict, value)
        sequence, policy_dict, value = self.examples[idx]
        
        # Sequence is already list of 64 integers (from board_to_sequence)
        # No padding needed for fixed board size
        
        policy_target = np.zeros(4096, dtype=np.float32)
        for move_idx, prob in policy_dict.items():
            if move_idx < 4096:
                policy_target[move_idx] = prob
                
        return torch.tensor(sequence, dtype=torch.long), policy_target, np.array([value], dtype=np.float32)

class Trainer:
    def __init__(self, device=None):
        self.device = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Transformer Trainer using: {self.device}")
        
        self.model = ChessTransformer().to(self.device)
        if os.path.exists("transformer_model.pth"):
            try:
                self.model.load_state_dict(torch.load("transformer_model.pth", weights_only=True))
                print("Loaded existing transformer model.")
            except:
                pass
                
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.0001) # Lower LR for Transformers
        self.mcts_sims = 200 # Increased to 200 for smarter play
        self.games_per_iteration = 10 # Increased to 10 for better stability
        self.batch_size = 32
        
        # Resume game count
        self.total_games_played = 0
        if os.path.exists("games.csv"):
            with open("games.csv", "r") as f:
                self.total_games_played = sum(1 for line in f) - 1 # Minus header
            if self.total_games_played < 0: self.total_games_played = 0
        print(f"Resuming from Game {self.total_games_played}")

    def execute_episode(self, game_idx=0, total_games=0):
        board = chess.Board()
        examples = []
        mcts = MCTS(self.model, self.device)
        
        # Try to clean up previous outputs if using IPython
        try:
            from IPython.display import clear_output, display, SVG
        except ImportError:
            pass

        while not board.is_game_over():
            if len(board.move_stack) > 200: break # Prevent infinite length
            
            # Save Live State for Spectator
            try:
                # Calculate Evaluation for the BAR
                current_seq = board_to_sequence(board)
                seq_tensor = torch.tensor([current_seq], dtype=torch.long, device=self.device)
                self.model.eval()
                with torch.no_grad():
                     _, val_tensor = self.model(seq_tensor)
                
                # Hybrid Eval for consistency with MCTS
                nn_val = val_tensor.item() # Relative (Good for mover)
                
                # We want the Bar to always show "White Advantage".
                # If it's Black's turn, nn_val=+1 means Black wins (White -1).
                if board.turn == chess.BLACK:
                    nn_val = -nn_val
                
                mat_score = get_material_score(board) # Absolute (White - Black)
                mat_val = np.tanh(mat_score / 3.0) 
                
                # 50/50 blend (or 30/70 to match training)
                eval_score = (0.3 * nn_val) + (0.7 * mat_val)
                
                with open("live.fen", "w") as f:
                    f.write(board.fen())
                    # Also write last move for highlighting
                    if board.move_stack:
                        f.write(f"\n{board.peek().uci()}")
                    else:
                        f.write("\nNone")
                        
                    # Write Eval Score (Line 3)
                    f.write(f"\n{eval_score:.4f}")
            except Exception as e:
                pass
            
            # VISUALIZATION
            # Only visualize every move (or every few moves)
            # VISUALIZATION
            # Only visualize every move
            # Strict check: Are we in a notebook (ZMQInteractiveShell)?
            try:
                # Check for explicit Notebook environment
                is_notebook = False
                try:
                    from IPython import get_ipython
                    if get_ipython().__class__.__name__ == 'ZMQInteractiveShell':
                        is_notebook = True
                except:
                    pass

                if is_notebook:
                    clear_output(wait=True)
                    display(SVG(chess.svg.board(board, size=300)))
                    print(f"Game {game_idx}/{total_games} | Move {len(board.move_stack)}")
                else:
                    # Console fallback
                    print(f"\rGame {game_idx}/{total_games} | Move {len(board.move_stack)}", end="", flush=True)
            except Exception:
                pass

            root = mcts.search(board, simulations=self.mcts_sims)
            # Temperature schedule: Explore for first 80 moves, then exploit
            # Changed 1.0 -> 0.7 to make play sharper and less random
            temp = 0.7 if len(board.move_stack) < 80 else 0.5

            moves, probs = mcts.get_action_probs(root, temperature=temp)
            
            policy_dict = {m: p for m, p in zip(moves, probs)}
            
            # Store SEQUENCE
            current_seq = board_to_sequence(board)
            examples.append([current_seq, policy_dict, None])
            
            choice_idx = np.random.choice(len(moves), p=probs)
            best_move_idx = moves[choice_idx]
            
            found_move = None
            for m in board.legal_moves:
                if encode_move(m) == best_move_idx:
                    found_move = m
                    break
            
            if found_move:
                board.push(found_move)
            else:
                break

        print(f"\rGame {game_idx}/{total_games} Finished. Result: {board.result()}                   ")
        
        # Save to CSV
        try:
            with open("games.csv", "a", encoding="utf-8") as f:
                # Format: GameID, Result, Moves(SAN)
                moves_san = [board.move_stack[i] for i in range(len(board.move_stack))]
                # Reconstruct SAN is hard without board replay, but board has move_stack. 
                # board.move_stack contains Move objects. Getting SAN requires board history.
                # Simplest is just UCI string space separated.
                moves_str = " ".join([m.uci() for m in board.move_stack])
                f.write(f"{game_idx},{board.result()},{moves_str}\n")
        except Exception as e:
            print(f"CSV Save Error: {e}")

        result = board.result()
        material_score = get_material_score(board) # positive means White has advantage
        
        # Base Reward
        if result == '1-0': 
            base_reward = 1.0
        elif result == '0-1': 
            base_reward = -1.0
        else: 
            # Draw Scenarios
            if board.can_claim_threefold_repetition():
                # Massive penalty for repetition to stop "cowardly" early draws
                base_reward = -0.5 
            else:
                # Standard draw penalty (Stalemate, insufficient material, 50-move)
                base_reward = -0.1 
            
        # Add small material incentive (e.g. 0.01 per pawn advantage)
        # We clamp it to avoid overshadowing the actual result
        # Max material diff is roughly 39. 39 * 0.01 = 0.39.
        # So a Win (1.0) is still better than capturing a Queen (0.09) but losing (-1).
        
        final_score_white = base_reward + (material_score * 0.01)
        
        # Clip to [-1, 1] range roughly, or just let it float. Tanh output is [-1,1].
        # Let's clip for stability? MCTS expects bounds usually.
        # But for AlphaZero MCTS, value is just expectation.
        # Let's keep it simple.
            
        processed = []
        for i, ex in enumerate(examples):
            # Perspective: 1 if White's turn, -1 if Black's turn
            # White's Move (Event 0, 2, 4...) -> Matches White's reward
            # Black's Move (Event 1, 3, 5...) -> Matches Black's reward (inverse of White)
            
            perspective = 1 if (i % 2 == 0) else -1
            
            # The value target should be from the perspective of the player who just moved?
            # No, usually it's "Value of the state for the current player".
            # If state is White to move, and White wins, Value = +1.
            # Ex[2] is the label for the board state `ex[0]`.
            # If ex[0] is Start Pos (White to move), and White eventually wins, Label = 1.2
            # If White wins, and it's Black's turn (perspective -1), Label = -1.2
            
            ex[2] = final_score_white * perspective
            processed.append(ex)
        return processed

    def train(self, iterations=1):
        for i in range(iterations):
            print(f"Iteration {i+1}")
            examples = []
            for g in range(self.games_per_iteration):
                current_game_idx = self.total_games_played + g + 1
                examples.extend(self.execute_episode(current_game_idx, self.games_per_iteration))
            
            self.total_games_played += self.games_per_iteration
                
            dataset = ChessDataset(examples)
            dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
            
            self.model.train()
            total_loss = 0
            for seq, p_target, v_target in tqdm(dataloader):
                seq, p_target, v_target = seq.to(self.device), p_target.to(self.device), v_target.to(self.device)
                
                self.optimizer.zero_grad()
                p_out, v_out = self.model(seq)
                
                v_loss = F.mse_loss(v_out.view(-1), v_target.view(-1))
                p_loss = -torch.sum(p_target * F.log_softmax(p_out, dim=1)) / seq.size(0)
                
                loss = v_loss + p_loss
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()
                
            avg_loss = total_loss / len(dataloader)
            print(f"Avg Loss: {avg_loss:.4f}")
            
            # Save Loss to CSV
            try:
                with open("loss.csv", "a") as f:
                    f.write(f"{self.total_games_played},{avg_loss:.4f}\n")
            except:
                pass

            torch.save(self.model.state_dict(), "transformer_model.pth")

if __name__ == "__main__":
    t = Trainer()
    t.train(10)


## 5. Run

In [ ]:
from trainer import Trainer
t = Trainer()
t.train(20)